In [1]:
import sys
sys.path.append("..")
import numpy as np
from sklearn.model_selection import train_test_split
from config import *
import pandas as pd
from Composition.pipelines import *
from Composition.run_experiment import *
from Composition.query_composition import *
import h5py
import numpy as np

In [2]:
# --------------------------------------------------
# Load data
# --------------------------------------------------
balanced_ids = np.load(balanced_ids, allow_pickle=True)
ids = np.load(ids_gaussian_new, allow_pickle=True)
compositions = np.load(comp_path, allow_pickle=True)
crystal_systems = np.load(cs_path , allow_pickle=True)
labels = np.load(sg_path, allow_pickle=True)
# --------------------------------------------------
# Align balanced dataset
# --------------------------------------------------
id_to_index = {id_: i for i, id_ in enumerate(ids)}

balanced_indices = np.array(
    [id_to_index[id_] for id_ in balanced_ids],
    dtype=np.int64
)

compositions_balanced = compositions[balanced_indices]
ids_balanced = ids[balanced_indices]
labels_balanced = labels[balanced_indices]
balanced_cs = crystal_systems[balanced_indices]

# --------------------------------------------------
# Remove duplicate compositions
# --------------------------------------------------

_, unique_indices = np.unique(
    compositions_balanced,
    return_index=True
)

compositions_balanced = compositions_balanced[unique_indices]
ids_balanced = ids_balanced[unique_indices]
labels_balanced = labels_balanced[unique_indices]
balanced_cs = balanced_cs[unique_indices]

# --------------------------------------------------
# Build metadata dictionary
# --------------------------------------------------

id_to_metadata = {
    real_id: {
        "composition": comp,
        "space_group": int(sg),
        "crystal_system": cs
    }
    for real_id, comp, sg, cs in zip(
        ids_balanced,
        compositions_balanced,
        labels_balanced,
        balanced_cs
    )
}

In [9]:
import pickle

with open("balanced_metadata.pkl", "wb") as f:
    pickle.dump(id_to_metadata, f)

print("Metadata saved successfully.")


Metadata saved successfully.


In [3]:
# --------------------------------------------------
# ELEMENT ENCODER
# --------------------------------------------------
result_compare = os.path.join(SAVE_EMBEDDING, "result")
elem_encoder = QwenElementEncoder()
X_elem = normalize(elem_encoder.encode(compositions_balanced))
np.save(ELEMENT_EMBEDDINGS_FULL, X_elem)

[ElementEncoder] No cache at qwen_element_cache.pkl


In [23]:
X_elem = np.load(ELEMENT_EMBEDDINGS_FULL)
search_engine = ElementSearchEngine(
    ids=ids_balanced,
    id_to_metadata=id_to_metadata,
    embeddings=X_elem)

[ElementEncoder] No cache at qwen_element_cache.pkl


In [11]:
results = search_engine.query("LiFePO4", top_k=5)
results

,rank,id,composition,score,space_group,crystal_system
0,1,1011090_SG_62,LiFePO4,1.0000,62,orthorhombic
1,2,mp-1177129,Li5Fe6P5O24,0.9998,1,triclinic
2,3,mp-1177121,Li7Fe4(PO4)6,0.9996,1,triclinic
3,4,mp-758654,Li3Fe2(PO4)3,0.9996,1,triclinic
4,5,mp-1982622,Li2FeP2O7,0.9994,1,triclinic


In [5]:
results = search_engine.query("SrTiO3", top_k=5)
results

,rank,id,composition,score,space_group,crystal_system
0,1,mp-5229,SrTiO3,1.0000,221,cubic
1,2,1521815_SG_221,Sr0.96Ca0.04Ti1O3,1.0000,221,cubic
2,3,mp-675134,Sr5Ti5O13,0.9997,2,triclinic
3,4,mp-1201432,Sr15Ti23O61,0.9995,10,monoclinic
4,5,mp-1218575,Sr5Ti4FeO15,0.9995,12,monoclinic


### VAE Dataset

In [24]:
def safe_decode(arr):
    return np.array([
        x.decode("utf-8") if isinstance(x, bytes) else x
        for x in arr
    ])

# ---- Load everything once ----
with h5py.File(external_test_dataset, "r") as f:
    compositions = safe_decode(f["compositions"][:])
    labels = f["labels"][:]                  # space groups (int)
    crystal_systems = safe_decode(f["crystalsystems"][:])

print("Total samples:", len(compositions))
print("Unique compositions:", np.unique(compositions)[:20])

# ---- Build ground truth dictionary ----
ground_truth = {
    comp: {
        "space_group": int(sg),
        "crystal_system": cs
    }
    for comp, sg, cs in zip(compositions, labels, crystal_systems)
}

Total samples: 465
Unique compositions: ['Al' 'Al0.24Co1.76' 'Al0.4Co0.6Ni3' 'Al0.58Co0.76Ni0.66' 'Al0.8Co1.2'
 'Al13Co4' 'Al148.204Co50.1' 'Al18Co5Ni3' 'Al1Ni0.89' 'Al24.2Co7.04Ni0.96'
 'Al3Ni' 'Al3Ni2' 'Al3Ni5' 'Al4Ni3' 'Al5Co2' 'Al70.8Co24'
 'Al72Co16.008Ni7.992' 'Al84Co31' 'Al9Co2' 'AlNi']


In [25]:
results = search_engine.query("Al0.24Co1.76", top_k=5)
results 

,rank,id,composition,score,space_group,crystal_system
0,1,1523557_SG_225,Al0.52Co3.48,1.0000,225,cubic
1,2,mp-1228933,AlCo4,0.9983,166,trigonal
2,3,mp-1188126,Pr2Al2Co15,0.9979,166,trigonal
3,4,mp-1220026,PrErCo17,0.9977,160,trigonal
4,5,mp-2531,Er2Co17,0.9975,194,hexagonal


In [8]:
results = search_engine.query("Al0.24Co1.76", top_k=5)
results 

,rank,id,composition,score,space_group,crystal_system
0,1,1523557_SG_225,Al0.52Co3.48,1.0000,225,cubic
1,2,mp-1228933,AlCo4,0.9983,166,trigonal
2,3,mp-1188126,Pr2Al2Co15,0.9979,166,trigonal
3,4,mp-1220026,PrErCo17,0.9977,160,trigonal
4,5,mp-2531,Er2Co17,0.9975,194,hexagonal


In [26]:
query_comp = "Al0.24Co1.76"

results = search_engine.query(query_comp, top_k=5)

true_info = ground_truth.get(query_comp)

print("Query:", query_comp)
print("True SG:", true_info["space_group"])
print("True Crystal System:", true_info["crystal_system"])
print("\nTop Retrieved Results:\n")

print(results)


Query: Al0.24Co1.76
True SG: 194
True Crystal System: hexagonal

Top Retrieved Results:

   rank              id   composition   score  space_group crystal_system
0     1  1523557_SG_225  Al0.52Co3.48  1.0000          225          cubic
1     2      mp-1228933         AlCo4  0.9983          166       trigonal
2     3      mp-1188126    Pr2Al2Co15  0.9979          166       trigonal
3     4      mp-1220026      PrErCo17  0.9977          160       trigonal
4     5         mp-2531       Er2Co17  0.9975          194      hexagonal


In [11]:
query_comp = "Al4Ni3"

results = search_engine.query(query_comp, top_k=5)

true_info = ground_truth.get(query_comp)

print("Query:", query_comp)
print("True SG:", true_info["space_group"])
print("True Crystal System:", true_info["crystal_system"])
print("\nTop Retrieved Results:\n")

print(results)

Query: Al4Ni3
True SG: 230
True Crystal System: cubic

Top Retrieved Results:

   rank          id   composition   score  space_group crystal_system
0     1    mp-16515        Al4Ni3  1.0000          230          cubic
1     2     mp-1057        Al3Ni2  0.9997          164       trigonal
2     3  mp-1228868          AlNi  0.9981          123     tetragonal
3     4  mp-1192626  Th2(Al3Ni2)5  0.9978           71   orthorhombic
4     5  mp-1228186      Al4Ni3Pt  0.9959          123     tetragonal


In [14]:
# ---- Load everything once ----
with h5py.File(external_test_ruff_dataset, "r") as f:
    compositions = safe_decode(f["compositions"][:])
    labels = f["labels"][:]                  # space groups (int)
    crystal_systems = safe_decode(f["crystalsystems"][:])

print("Total samples:", len(compositions))
print("Unique compositions:", np.unique(compositions))

# ---- Build ground truth dictionary ----
ground_truth = {
    comp: {
        "space_group": int(sg),
        "crystal_system": cs
    }
    for comp, sg, cs in zip(compositions, labels, crystal_systems)
}

Total samples: 1305
Unique compositions: ['Ag3Te2Au' 'AgTe4Au' 'B' 'BaFe(Si2O5)2' 'BaSi2O5' 'C' 'C4Br(NO2)2'
 'Ca24Al16Si16.008O96' 'Ca2Al2FeSi3O13' 'Ca3(AsO9)2' 'CaAl3PSO14' 'CaF2'
 'CaMg(CO3)2' 'CaU2(SiO6)2' 'Fe(SbS2)2' 'Fe1.132Co0.108Ni2.756' 'Fe2CuS3'
 'Fe4.2Si1.5O12.003' 'FeCu5S4' 'FeCuO2' 'FeS2' 'FeSbS' 'HgS'
 'K2Ca2Mg(S2O9)2' 'KAl3(SO7)2' 'KNa3Al4(SiO4)4' 'KSO3'
 'Li0.15Al0.15Si2.85O6' 'Li0.465Al0.465Si2.535O6' 'Li0.75Al0.75Si2.25O6'
 'Li0.99Al0.99Si2.01O6' 'Mg2Al3B2HO10' 'Mg3(SiO3)4'
 'Mg3.68Al4.2Fe0.06B4O16' 'MgAlBO4'
 'Na0.36Ca2.464Mg3.608Ti0.08Al1.368Fe0.84Si7.28O24' 'Na6S2ClO8F'
 'NaBe4SbO7' 'NaBeSi3O8' 'NaCl' 'NaH4ClO2' 'NaMg3Al6Si6B3O31' 'NiTe2'
 'PbS' 'PbSe' 'RbSO3' 'Sb2O3' 'Si(NF3)2' 'Si4H31.99968N8F24' 'SiC' 'SiO2'
 'SrAl3PSO14' 'SrF2' 'TePb' 'ThSiO4' 'USiO4'
 'Zn0.04Fe1.24Cu2.12Sn0.2Sb0.04Te0.08Mo0.36As0.12S2.84' 'ZnS']


In [15]:
import pickle

with open("balanced_metadata.pkl", "rb") as f:
    saved_meta = pickle.load(f)

print("Loaded metadata entries:", len(saved_meta))
saved_compositions = {
    v["composition"] for v in saved_meta.values()
}

print("Unique compositions in saved metadata:", len(saved_compositions))
unique_external = set(np.unique(compositions))

print("Unique compositions in external dataset:", len(unique_external))


Loaded metadata entries: 49839
Unique compositions in saved metadata: 49839
Unique compositions in external dataset: 58


In [ ]:
query_comp = "Ag3Te2Au"

results = search_engine.query(query_comp, top_k=5)

true_info = ground_truth.get(query_comp)

print("Query:", query_comp)
print("True SG:", true_info["space_group"])
print("True Crystal System:", true_info["crystal_system"])
print("\nTop Retrieved Results:\n")

results

Query: Ag3Te2Au
True SG: 214
True Crystal System: cubic

Top Retrieved Results:

   rank              id      composition   score  space_group crystal_system
0     1  1509990_SG_216  Si4Ag32.016Te24  0.9939          216          cubic
1     2  1509974_SG_216   Ag31.68Ge4Te24  0.9919          216          cubic
2     3       mp-676168         Ag8GeTe6  0.9919            1      triclinic
3     4       mp-685196      Tl2Ag16Te11  0.9903            1      triclinic
4     5  1509919_SG_136         CsAg5Te3  0.9897          136     tetragonal


In [16]:
query_comp = "MgAlBO4"

results = search_engine.query(query_comp, top_k=5)

true_info = ground_truth.get(query_comp)

print("Query:", query_comp)
print("True SG:", true_info["space_group"])
print("True Crystal System:", true_info["crystal_system"])
print("\nTop Retrieved Results:\n")

results

Query: MgAlBO4
True SG: 62
True Crystal System: orthorhombic

Top Retrieved Results:



,rank,id,composition,score,space_group,crystal_system
0,1,1533961_SG_62,AlBPbO4,0.9982,62,orthorhombic
1,2,mp-8110,AlBO3,0.9967,167,trigonal
2,3,1538311_SG_62,Mg5.32Ti1.36Al0.84Fe0.48B4O16,0.9962,62,orthorhombic
3,4,mp-1202393,UAl4B3O14,0.9961,11,monoclinic
4,5,mp-1182901,Al6B5O18,0.9961,176,hexagonal
